# Generate Parameter Ensembles
This script can create FATES parameter ensembles for a min/max one-at-a-time (OAAT) or latin hypercube (LH) experiments.

The main information required are a default FATES parameter file, an excel file with information about any parameters to be calibrated, and a list of parameters to include in the ensemble. 



In [1]:
import os
import pandas as pd
import xarray as xr
import fates_calibration_library.parameter_generation as param

In [5]:
# top directory
param_dir = '/glade/work/afoster/FATES_calibration/parameter_files'

# default parameter file
fates_param_name = "fates_params_default_sci.1.85.1_api.40.0.0_crops.nc"

# excel file with information about parameters
param_list_name = "param_list_sci.1.85.1_api.40.0.0.xls"

# list of parameters to include in OAAT ensemble
oaat_params_file = 'oaat_params.csv'

# list of parameters to include in LH ensemble
lh_params_file = 'lh_params.csv'

# directory to place OAAT files
oaat_dir = os.path.join(param_dir, 'fates_oaat')

# directory to place LH files
lh_dir = os.path.join(param_dir, 'fates_lh')

In [6]:
# get files
param_list_file = os.path.join(param_dir, param_list_name)
default_param_data = xr.open_dataset(os.path.join(param_dir, fates_param_name))
param_dat = param.get_param_dictionary(param_list_file)

In [7]:
oaat = False
lh = True

In [5]:
if oaat:
    # get list of parameters for OAAT experiment
    oaat_params = pd.read_csv(os.path.join(param_dir, oaat_params_file))['fates_parameter_name'].values
    
    # oaat ensemble
    param.create_oaat_param_ensemble(param_dat, oaat_params, default_param_data,
                                     oaat_dir, 'FATES_OAAT')
    oaat_key = pd.read_csv(os.path.join(oaat_dir, 'fates_oaat_key.csv'))

In [8]:
if lh:
    # get list of parameters for LH experiment
    lh_params = pd.read_csv(os.path.join(param_dir, lh_params_file))['fates_parameter_name'].values

    # lh ensemble
    param.create_lh_param_ensemble(lh_params, 500, default_param_data,
                                   param_dat, lh_dir, 'FATES_LH')
    # check the key just in case
    lh_key = pd.read_csv(os.path.join(lh_dir, 'fates_lh_key.csv'), index_col=0)

In [19]:
pars = lh_key.columns

In [10]:
files = sorted([os.path.join(lh_dir, f) for f in os.listdir(lh_dir) if f.endswith('nc')])

In [14]:
params = xr.open_mfdataset(files[:20],
                          combine='nested',
                          concat_dim='ensemble',
                          autoclose=True)

In [31]:
import fates_calibration_library.plotting_functions as plotting
import importlib
importlib.reload(plotting)

<module 'fates_calibration_library.plotting_functions' from '/glade/work/afoster/FATES_calibration/fates_calibration_library/fates_calibration_library/plotting_functions.py'>

In [40]:
diff = params['fates_nonhydro_smpso'] - params['fates_nonhydro_smpsc']

In [52]:
par = 'fates_stoich_nitr'

In [58]:
plotting.plot_params(default_param_data, params, pars[20])

KeyError: "No variable named 'ensemble'. Variables on the dataset include ['fates_pftname', 'fates_alloc_storage_cushion', 'fates_alloc_store_priority_frac', 'fates_allom_agb1', 'fates_allom_agb2', ..., 'fates_trs_seedling_emerg_h2o_timescale', 'fates_trs_seedling_mdd_timescale', 'fates_trs_seedling_mort_par_timescale', 'fates_vai_top_bin_width', 'fates_vai_width_increase_factor']"

In [60]:
len(pars)

21